import torch

from fish_segmentation.dinov3 import (
    load_backbone,
    plot_regions_with_centers,
    run_zoom_detector,
)
from fish_segmentation.paths import find_repo_root, load_env
from fish_segmentation.video_io import load_sampled_frames

load_env()
ROOT = find_repo_root()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


In [ ]:
model_id = "facebook/dinov3-vitl16-pretrain-lvd1689m"
model, processor = load_backbone(model_id, device=device)


In [ ]:
video_path = ROOT / "data" / "test_media" / "dolphin_00.mp4"
frames = load_sampled_frames(video_path=str(video_path), n_frames=5)


In [ ]:
# coarse-to-fine: DINO at <=1024 view, zoom into interest zones from original
# pixels, re-detect at native res, merge duplicate points, return [0,1] coords
results = []
for frame in frames:
    detections = run_zoom_detector(
        image=frame,
        model=model,
        processor=processor,
        long_side=1024,
    )
    results.append((frame, detections))

regions_per_frame = sum(len(d) for _, d in results) / len(results)
print(regions_per_frame)


In [ ]:
# visualize merged points on the full-res frame
import numpy as np

import matplotlib.pyplot as plt

frame, detections = results[1]
W, H = frame.size
fig, ax = plt.subplots(1, 1, figsize=(12, 7))
ax.imshow(frame)
for det in detections:
    cx, cy, r = det["x"] * W, det["y"] * H, det["radius"] * max(W, H)
    ax.scatter(cx, cy, c="cyan", edgecolors="black", s=60, zorder=5)
    ax.add_patch(plt.Circle((cx, cy), r, color="cyan", fill=False, linestyle="--", linewidth=1.2, alpha=0.8))
ax.set_title(f"{len(detections)} objects (frame {results.index((frame, detections))})")
ax.axis("off")
plt.show()


In [ ]:
# visualize merged detections for all sampled frames (full-res, by level)
import matplotlib.pyplot as plt

LEVEL_COLORS = ["tab:blue", "tab:green", "tab:orange", "tab:red", "tab:purple"]

n = len(results)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), squeeze=False)
for ax, (frame, detections) in zip(axes[0], results):
    W, H = frame.size
    ax.imshow(frame)
    for det in detections:
        cx, cy, r = det["x"] * W, det["y"] * H, det["radius"] * max(W, H)
        col = LEVEL_COLORS[det["level"] % len(LEVEL_COLORS)]
        ax.scatter(cx, cy, c=col, edgecolors="black", s=60, zorder=5)
        ax.add_patch(plt.Circle((cx, cy), r, color=col, fill=False, linestyle=":", lw=1.2))
    ax.set_title(f"{len(detections)} objects @ L{max((d['level'] for d in detections), default=0)}")
    ax.axis("off")
plt.tight_layout()
plt.show()